# MDOF Simulator

Objectives:

Write a reusable routine that takes any mass matrix `M` and stiffness matrix `K`, returns the
natural frequencies and mode shapes, and visualizes them — this is the workhorse for the rest
of the summer.

Assemble `M` and `K` for a 3-5 DOF spring-mass chain.

Solve the generalized eigenproblem (`scipy.linalg.eigh`) for natural frequencies and mode
shapes; mass-normalize the modes.

Verify orthogonality numerically (modes should diagonalize `M` and `K`).

Reconstruct the FRF as a modal sum and confirm it matches the direct matrix-inversion FRF.

Animate the mode shapes (`matplotlib FuncAnimation`) so you can see each mode move.

# Imports

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.linalg import eigh

# The workhorse routine

`solve_modes(M, K)` takes any mass matrix `M` and stiffness matrix `K` (same shape, symmetric
positive definite) and returns:

- `wn` — natural frequencies [rad/s], sorted ascending
- `fn` — natural frequencies [Hz]
- `Phi` — mass-normalized mode shape matrix, one column per mode (`Phi[:, i]` is mode i)

`scipy.linalg.eigh(K, M)` solves the generalized eigenproblem `K phi = omega^2 M phi` directly
(it's built for symmetric/Hermitian generalized problems, so no manual `M^-1 K` products or
eigenvector sorting are needed — `eigh` returns eigenvalues already sorted ascending).
Mass-normalizing means scaling each column so `phi_i^T M phi_i = 1`, which is what makes the
modes diagonalize `M` to the identity and `K` to `diag(omega_i^2)` (see the orthogonality
check below), and is the standard convention for modal superposition / FRF reconstruction.

In [2]:
def solve_modes(M, K):
    """Solve the generalized eigenproblem K phi = w^2 M phi.

    Returns
    -------
    wn  : (n,) natural frequencies [rad/s], ascending
    fn  : (n,) natural frequencies [Hz]
    Phi : (n, n) mass-normalized mode shapes, Phi[:, i] is mode i
    """
    eigvals, eigvecs = eigh(K, M)          # eigh handles the generalized symmetric problem
    eigvals = np.clip(eigvals, 0, None)    # guard against tiny negative numerical noise
    wn = np.sqrt(eigvals)
    fn = wn / (2 * np.pi)

    # eigh's eigenvectors already satisfy Phi^T M Phi = I when M is fed as the RHS matrix,
    # but we mass-normalize explicitly so the routine is correct even if that guarantee
    # is ever relied on loosely elsewhere.
    Phi = eigvecs.copy()
    for i in range(Phi.shape[1]):
        m_i = Phi[:, i] @ M @ Phi[:, i]
        Phi[:, i] /= np.sqrt(m_i)

    return wn, fn, Phi

# Assembling M and K for a spring-mass chain

A chain of `n` masses connected in series by springs, fixed at both ends:

```
wall --k0-- m0 --k1-- m1 --k2-- ... --kn-1-- m(n-1) --kn-- wall
```

Each mass only talks to its immediate neighbors, so `K` is tridiagonal: diagonal entries are
the sum of the two springs touching that mass, off-diagonal entries are `-k` for the shared
spring. `M` is diagonal since the masses aren't coupled directly.

In [3]:
def chain_matrices(masses, stiffnesses):
    """Build M, K for an n-DOF spring-mass chain fixed at both ends.

    masses       : length-n array of point masses
    stiffnesses  : length-(n+1) array of spring constants, wall-m0-m1-...-wall
    """
    n = len(masses)
    assert len(stiffnesses) == n + 1, "need n+1 springs for n masses fixed at both ends"

    M = np.diag(masses).astype(float)

    K = np.zeros((n, n))
    for i in range(n):
        K[i, i] = stiffnesses[i] + stiffnesses[i + 1]
    for i in range(n - 1):
        K[i, i + 1] = K[i + 1, i] = -stiffnesses[i + 1]

    return M, K


# 4-DOF example chain
masses = np.array([1.0, 1.5, 1.0, 2.0])          # [kg]
stiffnesses = np.array([400.0, 300.0, 300.0, 250.0, 200.0])  # [N/m], n+1 springs

M, K = chain_matrices(masses, stiffnesses)
print("M =\n", M)
print("K =\n", K)

M =
 [[1.  0.  0.  0. ]
 [0.  1.5 0.  0. ]
 [0.  0.  1.  0. ]
 [0.  0.  0.  2. ]]
K =
 [[ 700. -300.    0.    0.]
 [-300.  600. -300.    0.]
 [   0. -300.  550. -250.]
 [   0.    0. -250.  450.]]


# Solve for modes

In [4]:
wn, fn, Phi = solve_modes(M, K)

print("Natural frequencies:")
for i, (w, f) in enumerate(zip(wn, fn)):
    print(f"  mode {i+1}: wn = {w:8.3f} rad/s   fn = {f:7.3f} Hz")

print("\nMass-normalized mode shapes (columns):")
print(Phi)

Natural frequencies:
  mode 1: wn =    8.732 rad/s   fn =   1.390 Hz
  mode 2: wn =   15.545 rad/s   fn =   2.474 Hz
  mode 3: wn =   25.649 rad/s   fn =   4.082 Hz
  mode 4: wn =   29.987 rad/s   fn =   4.773 Hz

Mass-normalized mode shapes (columns):
[[-0.21829783 -0.32435394 -0.6134641  -0.6861504 ]
 [-0.45387826 -0.49556332 -0.08615758  0.45569184]
 [-0.51641973 -0.06801875  0.72454995 -0.45134408]
 [-0.43396342  0.51078051 -0.20923011  0.08367664]]


# Verify orthogonality

Mass-normalized modes should diagonalize `M` to the identity and `K` to `diag(omega_i^2)`:

`Phi^T M Phi = I`
`Phi^T K Phi = diag(wn^2)`

If the off-diagonal terms aren't ~0, something's wrong with the eigensolve or normalization.

In [5]:
Mr = Phi.T @ M @ Phi
Kr = Phi.T @ K @ Phi

print("Phi^T M Phi (should be identity):")
print(np.round(Mr, 8))

print("\nPhi^T K Phi (should be diag(wn^2)):")
print(np.round(Kr, 6))

print("\ndiag(wn^2) for comparison:")
print(np.round(wn**2, 6))

off_diag_M = Mr - np.diag(np.diag(Mr))
off_diag_K = Kr - np.diag(np.diag(Kr))
print(f"\nmax |off-diagonal| in Phi^T M Phi: {np.max(np.abs(off_diag_M)):.2e}")
print(f"max |off-diagonal| in Phi^T K Phi: {np.max(np.abs(off_diag_K)):.2e}")

Phi^T M Phi (should be identity):
[[ 1. -0. -0. -0.]
 [-0.  1. -0. -0.]
 [-0. -0.  1. -0.]
 [-0. -0. -0.  1.]]

Phi^T K Phi (should be diag(wn^2)):
[[ 76.249061  -0.        -0.        -0.      ]
 [ -0.       241.645788  -0.        -0.      ]
 [  0.        -0.       657.866686  -0.      ]
 [ -0.        -0.        -0.       899.238465]]

diag(wn^2) for comparison:
[ 76.249061 241.645788 657.866686 899.238465]

max |off-diagonal| in Phi^T M Phi: 2.79e-16
max |off-diagonal| in Phi^T K Phi: 1.25e-13


# FRF: modal sum vs. direct matrix inversion

The direct FRF comes from inverting the dynamic stiffness matrix at each frequency:

`H(w) = [K - w^2 M]^-1`

The modal sum reconstructs the same thing from the mode shapes and natural frequencies, with
no per-frequency matrix inversion, only a sum over modes:

`H(w) = sum_i  (phi_i phi_i^T) / (wn_i^2 - w^2)`

(undamped case — each mode contributes a rank-1 term that blows up at its own resonance). If
`solve_modes` is correct, these two should agree everywhere except exactly at resonance.

In [6]:
def frf_direct(w, M, K):
    """H(w) = [K - w^2 M]^-1, evaluated at each frequency in w. Returns (n, n, len(w))."""
    n = M.shape[0]
    H = np.zeros((n, n, len(w)), dtype=complex)
    for k, wk in enumerate(w):
        H[:, :, k] = np.linalg.inv(K - wk**2 * M)
    return H


def frf_modal(w, wn, Phi):
    """Undamped modal-sum reconstruction of the same FRF."""
    n = Phi.shape[0]
    H = np.zeros((n, n, len(w)), dtype=complex)
    for i in range(len(wn)):
        outer = np.outer(Phi[:, i], Phi[:, i])
        H += outer[:, :, None] / (wn[i]**2 - w[None, None, :]**2)
    return H


# sweep frequency, avoiding exact resonances so the direct inverse stays finite
w_sweep = np.linspace(0.1, 1.3 * wn[-1], 1500)
for wr in wn:
    w_sweep = w_sweep[np.abs(w_sweep - wr) > 0.5]

H_direct = frf_direct(w_sweep, M, K)
H_modal = frf_modal(w_sweep, wn, Phi)

diff = np.abs(H_direct - H_modal)
print(f"max |H_direct - H_modal| over the sweep: {np.max(diff):.2e}")
print(f"max |H_direct| over the sweep (for scale): {np.max(np.abs(H_direct)):.2e}")

max |H_direct - H_modal| over the sweep: 1.60e-16
max |H_direct| over the sweep (for scale): 3.18e-02


In [7]:
# visualize: drive point 0, measure at every DOF (H[i, 0, :] column)
drive = 0
fig, ax = plt.subplots(figsize=(9, 5))
for i in range(M.shape[0]):
    ax.semilogy(w_sweep, np.abs(H_direct[i, drive, :]), lw=2.5, alpha=0.5,
                label=f"direct, DOF {i}" if i == 0 else None, color=f"C{i}")
    ax.semilogy(w_sweep, np.abs(H_modal[i, drive, :]), "--", lw=1.2,
                label=f"modal sum, DOF {i}" if i == 0 else None, color="k")
for wr in wn:
    ax.axvline(wr, color="gray", ls=":", lw=1)
ax.set(title=f"FRF magnitude, driven at DOF {drive}: direct inverse vs. modal sum",
       xlabel="frequency [rad/s]", ylabel="|H| [m/N]")
ax.legend()
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig("mdof_frf_check.png", dpi=130)
print("Saved figure -> mdof_frf_check.png")

Saved figure -> mdof_frf_check.png


# Animate the mode shapes

Each mode is a pattern of relative motion at a single frequency: `x(t) = phi_i cos(wn_i t)`.
Plotting each mass's displacement about its equilibrium position over one period shows how the
DOFs move in and out of phase with each other for that mode.

In [8]:
def animate_mode(mode_index, wn, Phi, n_frames=60, amplitude_scale=0.3, spacing=1.0):
    """Animate a single mode shape: masses oscillate about equally-spaced equilibrium points."""
    n = Phi.shape[0]
    shape = Phi[:, mode_index]
    shape = shape / np.max(np.abs(shape))          # normalize for a consistent visual amplitude
    x_eq = np.arange(1, n + 1) * spacing            # equilibrium positions along the chain

    period = 2 * np.pi / wn[mode_index]
    t = np.linspace(0, period, n_frames, endpoint=False)

    fig, ax = plt.subplots(figsize=(7, 2.5))
    ax.set_xlim(0, (n + 1) * spacing)
    ax.set_ylim(-1.5, 1.5)
    ax.set_title(f"Mode {mode_index + 1}:  fn = {wn[mode_index] / (2*np.pi):.2f} Hz")
    ax.set_xlabel("equilibrium position along chain")
    ax.set_yticks([])
    ax.axhline(0, color="gray", lw=0.5)

    points, = ax.plot([], [], "o", ms=16, color=f"C{mode_index}")
    stems = [ax.plot([], [], "-", color="gray", lw=1)[0] for _ in range(n)]

    def init():
        points.set_data([], [])
        for s in stems:
            s.set_data([], [])
        return [points, *stems]

    def update(frame):
        y = amplitude_scale * shape * np.cos(wn[mode_index] * t[frame])
        points.set_data(x_eq, y)
        for i, s in enumerate(stems):
            s.set_data([x_eq[i], x_eq[i]], [0, y[i]])
        return [points, *stems]

    anim = FuncAnimation(fig, update, frames=n_frames, init_func=init,
                          interval=1000 / n_frames * period / (period / n_frames), blit=True)
    return fig, anim


fig, anim = animate_mode(0, wn, Phi)
anim.save("mdof_mode1.gif", writer=PillowWriter(fps=30))
plt.close(fig)
print("Saved animation -> mdof_mode1.gif")

Saved animation -> mdof_mode1.gif


In [9]:
# save an animation for every mode
for i in range(M.shape[0]):
    fig, anim = animate_mode(i, wn, Phi)
    anim.save(f"mdof_mode{i+1}.gif", writer=PillowWriter(fps=30))
    plt.close(fig)
    print(f"Saved animation -> mdof_mode{i+1}.gif")

Saved animation -> mdof_mode1.gif
Saved animation -> mdof_mode2.gif
Saved animation -> mdof_mode3.gif
Saved animation -> mdof_mode4.gif


# Superposition of modes: general free response

Modal superposition isn't just a trick for reconstructing the FRF — it's *the* statement that
the system's full free response is a sum of its modes. Because `Phi` diagonalizes both `M` and
`K`, the coordinate change `x = Phi @ q` decouples the equations of motion into `n` independent
single-DOF oscillators, one per mode:

`q_i(t) = q_i(0) cos(wn_i t) + (qdot_i(0) / wn_i) sin(wn_i t)`

Any initial condition `x(0), xdot(0)` maps to modal coordinates via `q(0) = Phi^T M x(0)` and
`qdot(0) = Phi^T M xdot(0)` (this is exactly `Phi^T M Phi = I` at work). The physical response
is then just the superposition `x(t) = Phi @ q(t) = sum_i phi_i * q_i(t)` — each mode
contributes its shape, scaled by its own independently-oscillating modal coordinate.

In [10]:
def free_response(t, wn, Phi, M, x0, v0):
    """General free response x(t) via modal superposition.

    t   : (n_t,) time samples
    x0  : (n,) initial displacement
    v0  : (n,) initial velocity
    Returns x(t) with shape (n, n_t).
    """
    q0 = Phi.T @ M @ x0
    qdot0 = Phi.T @ M @ v0

    # q_i(t) = q0_i cos(wn_i t) + (qdot0_i / wn_i) sin(wn_i t)
    q = (q0[:, None] * np.cos(wn[:, None] * t[None, :])
         + (qdot0 / wn)[:, None] * np.sin(wn[:, None] * t[None, :]))

    return Phi @ q


# initial condition: pluck DOF 0 only, everything else at rest
x0 = np.zeros(M.shape[0])
x0[0] = 1.0
v0 = np.zeros(M.shape[0])

t_check = np.linspace(0, 5, 400)
x_super = free_response(t_check, wn, Phi, M, x0, v0)

# sanity check: direct numerical integration of Mx'' + Kx = 0 should match the modal sum
from scipy.integrate import solve_ivp

def eom(t, state, M, K):
    n = M.shape[0]
    x, v = state[:n], state[n:]
    a = np.linalg.solve(M, -K @ x)
    return np.concatenate([v, a])

sol = solve_ivp(eom, (t_check[0], t_check[-1]), np.concatenate([x0, v0]),
                 t_eval=t_check, args=(M, K), rtol=1e-10, atol=1e-12)
x_direct = sol.y[:M.shape[0]]

print(f"max |x_super - x_direct|: {np.max(np.abs(x_super - x_direct)):.2e}")


max |x_super - x_direct|: 8.75e-10


In [11]:
fig, ax = plt.subplots(figsize=(9, 4))
for i in range(M.shape[0]):
    ax.plot(t_check, x_super[i], lw=2, label=f"DOF {i} (modal sum)")
    ax.plot(t_check, x_direct[i], "--", color="k", lw=1,
            label="direct integration" if i == 0 else None)
ax.set(title="Free response from plucking DOF 0: modal superposition vs. direct integration",
       xlabel="time [s]", ylabel="displacement [m]")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("mdof_superposition_check.png", dpi=130)
print("Saved figure -> mdof_superposition_check.png")


Saved figure -> mdof_superposition_check.png


# Animate the superposed response

Same idea as the single-mode animations, but now every mass moves according to the full
superposition `x(t) = Phi @ q(t)` for the plucked initial condition above — the combined
motion is generally *not* periodic in a simple way, since it's a sum of several incommensurate
frequencies.

In [12]:
def animate_superposition(t, x, wn, n_frames=150, spacing=1.0, amplitude_scale=1.0):
    """Animate the full superposed free response x(t), shape (n, n_t)."""
    n = x.shape[0]
    x_eq = np.arange(1, n + 1) * spacing

    frame_idx = np.linspace(0, x.shape[1] - 1, n_frames).astype(int)

    fig, ax = plt.subplots(figsize=(7, 2.5))
    ax.set_xlim(0, (n + 1) * spacing)
    ax.set_ylim(-1.5 * ampslitude_scale, 1.5 * amplitude_scale)
    ax.set_title("Superposed free response (all modes)")
    ax.set_xlabel("equilibrium position along chain")
    ax.set_yticks([])
    ax.axhline(0, color="gray", lw=0.5)

    points, = ax.plot([], [], "o", ms=16, color="C0")
    stems = [ax.plot([], [], "-", color="gray", lw=1)[0] for _ in range(n)]
    time_text = ax.text(0.02, 0.9, "", transform=ax.transAxes)

    def init():
        points.set_data([], [])
        for s in stems:
            s.set_data([], [])
        time_text.set_text("")
        return [points, *stems, time_text]

    def update(frame):
        k = frame_idx[frame]
        y = amplitude_scale * x[:, k]
        points.set_data(x_eq, y)
        for i, s in enumerate(stems):
            s.set_data([x_eq[i], x_eq[i]], [0, y[i]])
        time_text.set_text(f"t = {t[k]:.2f} s")
        return [points, *stems, time_text]

    anim = FuncAnimation(fig, update, frames=n_frames, init_func=init, blit=True)
    return fig, anim


fig, anim = animate_superposition(t_check, x_super, wn, amplitude_scale=1.0)
anim.save("mdof_superposition.gif", writer=PillowWriter(fps=30))
plt.close(fig)
print("Saved animation -> mdof_superposition.gif")


NameError: name 'ampslitude_scale' is not defined